In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import uuid
from datetime import datetime

CATALOG = "healthcare_medallion_dbw"

LANDING_BASE = "abfss://landing@tanvihealthstore2608.dfs.core.windows.net/"

BATCH_ID = str(uuid.uuid4())
PIPELINE_VERSION = "1.0"
ENVIRONMENT = "DEV"

In [0]:
sources = [
    {
        "source_id": "SRC_001",
        "source_name": "patients",
        "file_path": LANDING_BASE + "patients/patients.csv",
        "target_table": f"{CATALOG}.bronze.patients",
        "primary_key": "patient_id",
        "classification": "PHI"
    },
    {
        "source_id": "SRC_002",
        "source_name": "doctors",
        "file_path": LANDING_BASE + "doctors/doctors.csv",
        "target_table": f"{CATALOG}.bronze.doctors",
        "primary_key": "doctor_id",
        "classification": "PII"
    },
    {
        "source_id": "SRC_003",
        "source_name": "appointments",
        "file_path": LANDING_BASE + "appointments/appointments.csv",
        "target_table": f"{CATALOG}.bronze.appointments",
        "primary_key": "appointment_id",
        "classification": "PHI"
    },
    {
        "source_id": "SRC_004",
        "source_name": "treatments",
        "file_path": LANDING_BASE + "treatments/treatments.csv",
        "target_table": f"{CATALOG}.bronze.treatments",
        "primary_key": "treatment_id",
        "classification": "PHI"
    },
    {
    "source_id": "SRC_005",
    "source_name": "billing",
    "file_path": LANDING_BASE + "billings/billing.csv",
    "target_table": f"{CATALOG}.bronze.billing",
    "primary_key": "bill_id",
    "classification": "PHI"
}
]

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
    IntegerType,
    TimestampType
)

metadata_rows = []

for s in sources:
    metadata_rows.append((
        s["source_id"],
        s["source_name"],
        s["file_path"],
        "csv",
        s["target_table"],
        s["primary_key"],
        "Y",
        "FULL",
        None,
        "v1.0",
        "Healthcare Source System",
        s["classification"],
        1,
        0.10,
        "dataops@hospital.org",
        3,
        f"abfss://quarantine@tanvihealthstore2608.dfs.core.windows.net/{s['source_name']}/",
        "_ingestion_date",
        "06:00",
        None
    ))

metadata_schema = StructType([
    StructField("source_id", StringType(), True),
    StructField("source_name", StringType(), True),
    StructField("file_path", StringType(), True),
    StructField("file_format", StringType(), True),
    StructField("target_table", StringType(), True),
    StructField("primary_key", StringType(), True),
    StructField("active_flag", StringType(), True),
    StructField("load_type", StringType(), True),
    StructField("last_load_timestamp", TimestampType(), True),
    StructField("expected_schema_version", StringType(), True),
    StructField("source_system_owner", StringType(), True),
    StructField("data_classification", StringType(), True),
    StructField("row_count_threshold", LongType(), True),
    StructField("max_null_pct", DoubleType(), True),
    StructField("notification_email", StringType(), True),
    StructField("retry_count", IntegerType(), True),
    StructField("quarantine_path", StringType(), True),
    StructField("partition_column", StringType(), True),
    StructField("sla_cutoff_time", StringType(), True),
    StructField("dependency_source_ids", StringType(), True)
])

metadata_df = spark.createDataFrame(
    metadata_rows,
    schema=metadata_schema
)

metadata_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{CATALOG}.ops.metadata_config"
    )

print("Metadata config loaded successfully")

Metadata config loaded successfully


In [0]:
display(
    spark.table(
        f"{CATALOG}.ops.metadata_config"
    )
)

source_id,source_name,file_path,file_format,target_table,primary_key,active_flag,load_type,last_load_timestamp,expected_schema_version,source_system_owner,data_classification,row_count_threshold,max_null_pct,notification_email,retry_count,quarantine_path,partition_column,sla_cutoff_time,dependency_source_ids
SRC_001,patients,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/patients/patients.csv,csv,healthcare_medallion_dbw.bronze.patients,patient_id,Y,FULL,null,v1.0,Healthcare Source System,PHI,1,0.1,dataops@hospital.org,3,abfss://quarantine@tanvihealthstore2608.dfs.core.windows.net/patients/,_ingestion_date,06:00,null
SRC_002,doctors,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/doctors/doctors.csv,csv,healthcare_medallion_dbw.bronze.doctors,doctor_id,Y,FULL,null,v1.0,Healthcare Source System,PII,1,0.1,dataops@hospital.org,3,abfss://quarantine@tanvihealthstore2608.dfs.core.windows.net/doctors/,_ingestion_date,06:00,null
SRC_003,appointments,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/appointments/appointments.csv,csv,healthcare_medallion_dbw.bronze.appointments,appointment_id,Y,FULL,null,v1.0,Healthcare Source System,PHI,1,0.1,dataops@hospital.org,3,abfss://quarantine@tanvihealthstore2608.dfs.core.windows.net/appointments/,_ingestion_date,06:00,null
SRC_004,treatments,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/treatments/treatments.csv,csv,healthcare_medallion_dbw.bronze.treatments,treatment_id,Y,FULL,null,v1.0,Healthcare Source System,PHI,1,0.1,dataops@hospital.org,3,abfss://quarantine@tanvihealthstore2608.dfs.core.windows.net/treatments/,_ingestion_date,06:00,null
SRC_005,billing,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/billings/billing.csv,csv,healthcare_medallion_dbw.bronze.billing,bill_id,Y,FULL,null,v1.0,Healthcare Source System,PHI,1,0.1,dataops@hospital.org,3,abfss://quarantine@tanvihealthstore2608.dfs.core.windows.net/billing/,_ingestion_date,06:00,null


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
    IntegerType,
    BooleanType,
    TimestampType
)
import uuid
from datetime import datetime


def ingest_to_bronze(source):

    start_time = datetime.now()

    source_name = source["source_name"]
    file_path = source["file_path"]
    target_table = source["target_table"]

    try:

        raw_df = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(file_path)
        )

        rows_read = raw_df.count()

        business_columns = raw_df.columns

        bronze_df = (
            raw_df

            .withColumn(
                "_ingestion_timestamp",
                F.current_timestamp()
            )

            .withColumn(
                "_source_file_name",
                F.lit(file_path)
            )

            .withColumn(
                "_batch_id",
                F.lit(BATCH_ID)
            )

            .withColumn(
                "_layer",
                F.lit("BRONZE")
            )

            .withColumn(
                "_ingestion_date",
                F.current_date()
            )

            .withColumn(
                "_pipeline_version",
                F.lit(PIPELINE_VERSION)
            )

            .withColumn(
                "_source_system",
                F.lit(source_name)
            )

            .withColumn(
                "_record_hash",
                F.sha2(
                    F.concat_ws(
                        "||",
                        *[
                            F.coalesce(
                                F.col(c).cast("string"),
                                F.lit("")
                            )
                            for c in business_columns
                        ]
                    ),
                    256
                )
            )
        )

        duplicate_window = Window.partitionBy(
            "_record_hash"
        )

        bronze_df = (
            bronze_df

            .withColumn(
                "_duplicate_count",
                F.count("*").over(
                    duplicate_window
                )
            )

            .withColumn(
                "_is_duplicate",
                F.col("_duplicate_count") > 1
            )

            .drop("_duplicate_count")

            .withColumn(
                "_raw_row_number",
                F.monotonically_increasing_id()
            )
        )

        (
            bronze_df.write
            .format("delta")
            .mode("overwrite")
            .option(
                "overwriteSchema",
                "true"
            )
            .saveAsTable(
                target_table
            )
        )

        rows_written = bronze_df.count()

        end_time = datetime.now()

        duration = int(
            (end_time - start_time).total_seconds()
        )

        audit_schema = StructType([
            StructField("audit_id", StringType(), True),
            StructField("batch_id", StringType(), True),
            StructField("source_name", StringType(), True),
            StructField("layer", StringType(), True),
            StructField("pipeline_start_time", TimestampType(), True),
            StructField("pipeline_end_time", TimestampType(), True),
            StructField("rows_read", LongType(), True),
            StructField("rows_written", LongType(), True),
            StructField("rows_rejected", LongType(), True),
            StructField("status", StringType(), True),
            StructField("error_message", StringType(), True),
            StructField("triggered_by", StringType(), True),
            StructField("created_at", TimestampType(), True),
            StructField("pipeline_duration_secs", LongType(), True),
            StructField("notebook_name", StringType(), True),
            StructField("cluster_id", StringType(), True),
            StructField("spark_app_id", StringType(), True),
            StructField("rows_quarantined", LongType(), True),
            StructField("dq_score_avg", DoubleType(), True),
            StructField("schema_version", StringType(), True),
            StructField("environment", StringType(), True),
            StructField("retry_attempt", IntegerType(), True),
            StructField("data_classification", StringType(), True),
            StructField("sla_met", BooleanType(), True),
            StructField("downstream_notified", BooleanType(), True)
        ])

        audit_row = [(
            str(uuid.uuid4()),
            BATCH_ID,
            source_name,
            "BRONZE",
            start_time,
            end_time,
            rows_read,
            rows_written,
            0,
            "SUCCESS",
            None,
            "manual",
            datetime.now(),
            duration,
            "01_ingest_bronze",
            None,
            None,
            0,
            None,
            "v1.0",
            ENVIRONMENT,
            0,
            source["classification"],
            True,
            False
        )]

        audit_df = spark.createDataFrame(
            audit_row,
            schema=audit_schema
        )

        (
            audit_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(
                f"{CATALOG}.ops.pipeline_audit_log"
            )
        )

        print(
            f"{source_name} loaded successfully"
        )

    except Exception as e:

        print(
            f"{source_name} FAILED: {str(e)}"
        )

In [0]:
for source in sources:
    ingest_to_bronze(source)

patients loaded successfully
doctors loaded successfully
appointments loaded successfully
treatments loaded successfully
billing loaded successfully


In [0]:
display(
    spark.read.format("binaryFile")
    .load(
        "abfss://landing@tanvihealthstore2608.dfs.core.windows.net/billings/*"
    )
    .select("path")
)

path
abfss://landing@tanvihealthstore2608.dfs.core.windows.net/billings/billing.csv


In [0]:
spark.sql(
    f"SHOW TABLES IN {CATALOG}.bronze"
).show(truncate=False)

+--------+------------+-----------+
|database|tableName   |isTemporary|
+--------+------------+-----------+
|bronze  |appointments|false      |
|bronze  |billing     |false      |
|bronze  |doctors     |false      |
|bronze  |patients    |false      |
|bronze  |treatments  |false      |
+--------+------------+-----------+



In [0]:
for table in [
    "patients",
    "doctors",
    "appointments",
    "treatments",
    "billing"
]:
    print(
        table,
        spark.table(
            f"{CATALOG}.bronze.{table}"
        ).count()
    )

patients 50
doctors 10
appointments 200
treatments 200
billing 200


In [0]:
display(
    spark.table(
        f"{CATALOG}.bronze.patients"
    )
)

patient_id,first_name,last_name,gender,date_of_birth,contact_number,address,registration_date,insurance_provider,insurance_number,email,_ingestion_timestamp,_source_file_name,_batch_id,_layer,_ingestion_date,_pipeline_version,_source_system,_record_hash,_is_duplicate,_raw_row_number
P030,Emily,Moore,M,1964-12-23,6622318721,456 Oak Ave,2021-08-07,PulseSecure,INS250262,emily.moore@mail.com,2026-08-10T15:37:39.071Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/patients/patients.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,patients,02ad33d613e2e3a13c303e3b6ecdb312de17f26918589a17597ab36f10cc833b,false,0
P037,Robert,Williams,M,1999-02-05,8886800195,456 Oak Ave,2021-09-30,HealthIndia,INS319963,robert.williams@mail.com,2026-08-10T15:37:39.071Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/patients/patients.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,patients,0488141dacb9e95c7214d2fc26abb66b08e51c8917a78f76e1c480bdc11fb1e6,false,1
P020,Jane,Moore,F,2003-06-06,8158989953,789 Pine Rd,2022-04-03,MedCare Plus,INS276089,jane.moore@mail.com,2026-08-10T15:37:39.071Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/patients/patients.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,patients,07254aaaacc5e22d548525b4c563fdafaadd394189f0c8a670f4b3bca845fadc,false,2
P007,Alex,Johnson,F,1989-06-08,6278710077,789 Pine Rd,2021-12-25,MedCare Plus,INS465890,alex.johnson@mail.com,2026-08-10T15:37:39.071Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/patients/patients.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,patients,08d33f0571ac6307f0531f077f04e3d9f02ecf29ac066d29b293ba6b900f5ec4,false,3
P033,Michael,Wilson,F,1970-02-06,7923214041,789 Pine Rd,2023-09-06,MedCare Plus,INS544209,michael.wilson@mail.com,2026-08-10T15:37:39.071Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/patients/patients.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,patients,0e3c5f8840ba22f966e0e0770c80a349acecad423a2f4c2280a9206dcad0bb46,false,4
P025,Robert,Wilson,M,1966-08-14,7482069727,123 Elm St,2021-09-09,HealthIndia,INS833429,robert.wilson@mail.com,2026-08-10T15:37:39.071Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/patients/patients.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,patients,110fdecbaef8f34f703d682c7af0025d1ffa697868a536214580f0f1044b2b9b,false,5
P026,John,Taylor,M,2003-11-28,9900972256,123 Elm St,2021-05-13,MedCare Plus,INS598863,john.taylor@mail.com,2026-08-10T15:37:39.071Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/patients/patients.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,patients,151e32787fcbf595a0f0270949b60d75357581f6b90f731283479261e5c7b5bf,false,6
P038,David,Smith,M,1991-06-25,6347262390,789 Pine Rd,2021-04-19,MedCare Plus,INS580761,david.smith@mail.com,2026-08-10T15:37:39.071Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/patients/patients.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,patients,1594656a93d32fb4e8b981e2b8f0065fd2b5a9af30b638461e69570833ba6abe,false,7
P031,Robert,Miller,M,1987-01-14,8280346676,321 Maple Dr,2022-06-28,WellnessCorp,INS542905,robert.miller@mail.com,2026-08-10T15:37:39.071Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/patients/patients.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,patients,1afb9a46ba50bcf68bf52706e27e2af144b006399b82fcf8495d1fdbb244d33d,false,8
P039,Jane,Wilson,F,1950-12-12,9271131338,789 Pine Rd,2021-03-09,PulseSecure,INS348710,jane.wilson@mail.com,2026-08-10T15:37:39.071Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/patients/patients.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,patients,1bc238e6a4b37fbcc9a5fb9ddca84d10ce9364a6c39377b624c45bb6f64d85f1,false,9


In [0]:
display(
    spark.table(
        f"{CATALOG}.ops.pipeline_audit_log"
    )
    .orderBy(
        F.col("created_at").desc()
    )
)

audit_id,batch_id,source_name,layer,pipeline_start_time,pipeline_end_time,rows_read,rows_written,rows_rejected,status,error_message,triggered_by,created_at,pipeline_duration_secs,notebook_name,cluster_id,spark_app_id,rows_quarantined,dq_score_avg,schema_version,environment,retry_attempt,data_classification,sla_met,downstream_notified
b02c508b-80c3-4c55-b120-02cd7bc7d97c,f85645bc-e96c-4b26-a616-4a8fc95a69fa,billing,BRONZE,2026-08-10T15:37:54.974Z,2026-08-10T15:37:59.024Z,200,200,0,SUCCESS,null,manual,2026-08-10T15:37:59.024Z,4,01_ingest_bronze,null,null,0,null,v1.0,DEV,0,PHI,true,false
f7300c42-58af-4442-bd23-7d35ab2910d6,f85645bc-e96c-4b26-a616-4a8fc95a69fa,treatments,BRONZE,2026-08-10T15:37:50.546Z,2026-08-10T15:37:53.870Z,200,200,0,SUCCESS,null,manual,2026-08-10T15:37:53.870Z,3,01_ingest_bronze,null,null,0,null,v1.0,DEV,0,PHI,true,false
75c2fb13-ce06-41f6-8a35-001d7b0e7774,f85645bc-e96c-4b26-a616-4a8fc95a69fa,appointments,BRONZE,2026-08-10T15:37:46.118Z,2026-08-10T15:37:49.413Z,200,200,0,SUCCESS,null,manual,2026-08-10T15:37:49.413Z,3,01_ingest_bronze,null,null,0,null,v1.0,DEV,0,PHI,true,false
68ca627e-47f1-4bbb-850d-f0de4e253507,f85645bc-e96c-4b26-a616-4a8fc95a69fa,doctors,BRONZE,2026-08-10T15:37:41.673Z,2026-08-10T15:37:44.939Z,10,10,0,SUCCESS,null,manual,2026-08-10T15:37:44.939Z,3,01_ingest_bronze,null,null,0,null,v1.0,DEV,0,PII,true,false
1abe8311-e716-4824-892b-c320f0cfd2c4,f85645bc-e96c-4b26-a616-4a8fc95a69fa,patients,BRONZE,2026-08-10T15:37:36.230Z,2026-08-10T15:37:40.404Z,50,50,0,SUCCESS,null,manual,2026-08-10T15:37:40.404Z,4,01_ingest_bronze,null,null,0,null,v1.0,DEV,0,PHI,true,false
dfca462a-adf3-46f8-8f9f-cfde8e09c699,f85645bc-e96c-4b26-a616-4a8fc95a69fa,treatments,BRONZE,2026-08-10T15:10:32.451Z,2026-08-10T15:10:35.440Z,200,200,0,SUCCESS,null,manual,2026-08-10T15:10:35.440Z,2,01_ingest_bronze,null,null,0,null,v1.0,DEV,0,PHI,true,false
ea0495e6-0a34-40d3-bc73-a9f4ad9e87d8,f85645bc-e96c-4b26-a616-4a8fc95a69fa,appointments,BRONZE,2026-08-10T15:10:28.427Z,2026-08-10T15:10:31.285Z,200,200,0,SUCCESS,null,manual,2026-08-10T15:10:31.285Z,2,01_ingest_bronze,null,null,0,null,v1.0,DEV,0,PHI,true,false
5bd75744-3584-4901-8113-a8168faeb7d1,f85645bc-e96c-4b26-a616-4a8fc95a69fa,doctors,BRONZE,2026-08-10T15:10:24.409Z,2026-08-10T15:10:27.214Z,10,10,0,SUCCESS,null,manual,2026-08-10T15:10:27.214Z,2,01_ingest_bronze,null,null,0,null,v1.0,DEV,0,PII,true,false
d20cee83-3d20-4ae1-844b-541ae3e192a4,f85645bc-e96c-4b26-a616-4a8fc95a69fa,patients,BRONZE,2026-08-10T15:10:19.356Z,2026-08-10T15:10:22.946Z,50,50,0,SUCCESS,null,manual,2026-08-10T15:10:22.947Z,3,01_ingest_bronze,null,null,0,null,v1.0,DEV,0,PHI,true,false


In [0]:
display(
    dbutils.fs.ls(
        "abfss://landing@tanvihealthstore2608.dfs.core.windows.net/billings/"
    )
)

path,name,size,modificationTime
abfss://landing@tanvihealthstore2608.dfs.core.windows.net/billings/billing.csv,billing.csv,10018,1786369467000


In [0]:
billing_source = [
    s for s in sources
    if s["source_name"] == "billing"
][0]

ingest_to_bronze(billing_source)

billing loaded successfully


In [0]:
print(
    "Billing rows:",
    spark.table(
        f"{CATALOG}.bronze.billing"
    ).count()
)

Billing rows: 200


In [0]:
display(
    spark.table(
        f"{CATALOG}.bronze.billing"
    )
)

bill_id,patient_id,treatment_id,bill_date,amount,payment_method,payment_status,_ingestion_timestamp,_source_file_name,_batch_id,_layer,_ingestion_date,_pipeline_version,_source_system,_record_hash,_is_duplicate,_raw_row_number
B030,P026,T030,2023-08-29,1316.47,Credit Card,Pending,2026-08-10T16:09:50.712Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/billings/billing.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,billing,01879408a1c57f89676c5a6967ae9c6c1e3632bb45cba811469edc17d26c0225,false,0
B095,P007,T095,2023-05-09,2097.48,Cash,Failed,2026-08-10T16:09:50.712Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/billings/billing.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,billing,04335fb4f220f08ecda50a3316e83c5ce9a9eb0017fbaac332558754094b3658,false,1
B141,P041,T141,2023-06-15,3689.35,Insurance,Pending,2026-08-10T16:09:50.712Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/billings/billing.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,billing,055b9d1a43494c457f36f7671ebc0d886f19c8d0411ef91cff46ee6aec8d24ce,false,2
B107,P009,T107,2023-04-17,3512.69,Credit Card,Pending,2026-08-10T16:09:50.712Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/billings/billing.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,billing,06d74ae5afcb6762317f63abfc3df4dcf0457f9495ad4a1691573d9f3c447ff0,false,3
B169,P029,T169,2023-07-24,2313.41,Credit Card,Pending,2026-08-10T16:09:50.712Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/billings/billing.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,billing,0af4bbbcafb452728d42e912118d1cff5fba9dc2efe62556b665c4acc93b05ec,false,4
B075,P043,T075,2023-05-08,2735.45,Cash,Failed,2026-08-10T16:09:50.712Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/billings/billing.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,billing,0d3f3f88ebeaafabbfcbbafb88dd2d785dd8580e0f8d3f71a7edcad380a676a7,false,5
B032,P048,T032,2023-11-06,3690.71,Insurance,Paid,2026-08-10T16:09:50.712Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/billings/billing.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,billing,0db170eb6f13be8000187cea576467dde1ace559594c72dad13dc2cbe9a0da74,false,6
B178,P017,T178,2023-01-17,4652.41,Cash,Pending,2026-08-10T16:09:50.712Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/billings/billing.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,billing,0df0cadeb33373029bf1357e492418347dd8ca32fb6c979b7a3288e5a1a8adc8,false,7
B142,P019,T142,2023-11-01,662.72,Insurance,Paid,2026-08-10T16:09:50.712Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/billings/billing.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,billing,0e111bfcbf456b9588bbd4333882e83edfc5f0d251c7ae3a8c238acee9fcefe8,false,8
B144,P009,T144,2023-08-16,1684.01,Insurance,Failed,2026-08-10T16:09:50.712Z,abfss://landing@tanvihealthstore2608.dfs.core.windows.net/billings/billing.csv,f85645bc-e96c-4b26-a616-4a8fc95a69fa,BRONZE,2026-08-10,1.0,billing,0e85a321b623196903daca6d7dd1a005f2fd1c51f97e2cc55c778c6b80b67b87,false,9
